In [ ]:
import zipfile
import os

zip_file = 'Market-1501-v15.09.15.zip'

extracted_dir = 'raw_dataset/'

os.makedirs(extracted_dir, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extracted_dir)

extracted_contents = os.listdir(extracted_dir)
print("Contents of the extracted directory:", extracted_contents)

Contents of the extracted directory: ['Market-1501-v15.09.15']


In [ ]:
from PIL import Image, UnidentifiedImageError
import os
import numpy as np
from sklearn.model_selection import train_test_split

# Set the path to the extracted dataset
dataset_dir = 'raw_dataset/Market-1501-v15.09.15'

# Set the destination directories for preprocessed data
train_dir = 'dataset/market1501/train/'
test_dir = 'dataset/market1501/test/'

# Create the destination directories if they don't exist
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)

# Function to preprocess images
def preprocess_images(src_dir, dest_dir):
    for img_name in os.listdir(src_dir):
        img_path = os.path.join(src_dir, img_name)

        try:
            # Open the image
            img = Image.open(img_path)

            # Convert the image to RGB if it's not in that mode
            img = img.convert("RGB")

            # Resize the image to a consistent size (e.g., 128x64) with LANCZOS antialiasing
            img = img.resize((128, 64), Image.LANCZOS)

            # Convert the image to a NumPy array and normalize pixel values to [0, 1]
            img_array = np.array(img) / 255.0

            # Save the preprocessed image to the destination directory
            dest_path = os.path.join(dest_dir, img_name)
            np.save(dest_path, img_array)
        except (UnidentifiedImageError, OSError):
            # Skip files that cannot be identified as images
            print(f"Skipping {img_path} as it cannot be identified as an image.")

# Preprocess training images
train_src_dir = os.path.join(dataset_dir, 'bounding_box_train')
preprocess_images(train_src_dir, train_dir)

# Preprocess testing images
test_src_dir = os.path.join(dataset_dir, 'bounding_box_test')
preprocess_images(test_src_dir, test_dir)

# Optional: Split the training data into training and validation sets
train_images, val_images = train_test_split(os.listdir(train_dir), test_size=0.2, random_state=42)

# Set the destination directories for training and validation data
train_split_dir = 'dataset/market1501/train_split/'
val_split_dir = 'dataset/market1501/val_split/'

# Create the destination directories if they don't exist
os.makedirs(train_split_dir, exist_ok=True)
os.makedirs(val_split_dir, exist_ok=True)

# Move images to the appropriate directories
for img_name in train_images:
    src_path = os.path.join(train_dir, img_name)
    dest_path = os.path.join(train_split_dir, img_name)
    os.rename(src_path, dest_path)

for img_name in val_images:
    src_path = os.path.join(train_dir, img_name)
    dest_path = os.path.join(val_split_dir, img_name)
    os.rename(src_path, dest_path)


Skipping raw_dataset/Market-1501-v15.09.15/bounding_box_train/Thumbs.db as it cannot be identified as an image.
Skipping raw_dataset/Market-1501-v15.09.15/bounding_box_test/Thumbs.db as it cannot be identified as an image.


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import ToTensor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Define the ResNet-based model
class ResNetPersonReID(nn.Module):
    def __init__(self, num_classes):
        super(ResNetPersonReID, self).__init__()
        self.resnet = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

# Define a custom dataset class
class PersonReIDDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = os.listdir(data_dir)
        self.label_encoder = LabelEncoder()

        # Extract labels from file names (assuming file names are in the format "label_xxx.npy")
        self.labels = [int(file.split('_')[0]) for file in self.image_files]
        self.label_encoder.fit(self.labels)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_files[idx])
        img_array = np.load(img_path)
        label = self.label_encoder.transform([self.labels[idx]])[0]

        return {'image': img_array, 'label': label}

# Set random seed for reproducibility
torch.manual_seed(42)

# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_classes = 751  # Adjust based on your dataset
batch_size = 32
learning_rate = 0.001
num_epochs = 10

# Create model, loss function, and optimizer
model = ResNetPersonReID(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Create datasets and data loaders
train_dataset = PersonReIDDataset(train_split_dir)
val_dataset = PersonReIDDataset(val_split_dir)
test_dataset = PersonReIDDataset(test_dir)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Training loop
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Validation loop
model.eval()
val_predictions, val_true_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        val_predictions.extend(predictions.cpu().numpy())
        val_true_labels.extend(labels.cpu().numpy())

# Evaluate on validation set
val_accuracy = accuracy_score(val_true_labels, val_predictions)
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')

# Test loop
model.eval()
test_predictions, test_true_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        test_predictions.extend(predictions.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Evaluate on test set
test_accuracy = accuracy_score(test_true_labels, test_predictions)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')


Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /root/.cache/torch/hub/v0.10.0.zip
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 50.6MB/s]


RuntimeError: Given groups=1, weight of size [64, 3, 7, 7], expected input[32, 64, 128, 3] to have 3 channels, but got 64 channels instead

In [2]:
!pip install torch torchvision
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import ToTensor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from PIL import Image


# Define the ResNet-based model
class ResNetPersonReID(nn.Module):
    def __init__(self, num_classes):
        super(ResNetPersonReID, self).__init__()
        self.resnet = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18', pretrained=True)
        in_features = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.resnet(x)

# ... (code remains the same)

# Define a custom dataset class
class PersonReIDDataset(Dataset):
    def __init__(self, data_dir):
        self.data_dir = data_dir
        self.image_files = os.listdir(data_dir)
        self.label_encoder = LabelEncoder()

        # Extract labels from file names (assuming file names are in the format "label_xxx.npy")
        self.labels = [int(file.split('_')[0]) for file in self.image_files]
        self.label_encoder.fit(self.labels)

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.data_dir, self.image_files[idx])
        try:
            img = Image.open(img_path).convert('RGB')  # Convert to RGB format
        except IOError:
            # Handle non-image files appropriately (e.g., skip them)
            pass
        return img

# Set random seed for reproducibility
torch.manual_seed(42)

# Set device (use GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameters
num_classes = 751  # Adjust based on your dataset
batch_size = 3

# Create model, loss function, and optimizer
model = ResNetPersonReID(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Create datasets and data loaders
train_dataset = PersonReIDDataset(train_split_dir)
val_dataset = PersonReIDDataset(val_split_dir)
test_dataset = PersonReIDDataset(test_dir)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Training loop
for epoch in range(num_epochs):
    model.train()
    for batch in train_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Validation loop
model.eval()
val_predictions, val_true_labels = [], []
with torch.no_grad():
    for batch in val_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        val_predictions.extend(predictions.cpu().numpy())
        val_true_labels.extend(labels.cpu().numpy())

# Evaluate on validation set
val_accuracy = accuracy_score(val_true_labels, val_predictions)
print(f'Validation Accuracy: {val_accuracy * 100:.2f}%')

# Test loop
model.eval()
test_predictions, test_true_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        images, labels = batch['image'].to(device), batch['label'].to(device)
        outputs = model(images)
        _, predictions = torch.max(outputs, 1)
        test_predictions.extend(predictions.cpu().numpy())
        test_true_labels.extend(labels.cpu().numpy())

# Evaluate on test set
test_accuracy = accuracy_score(test_true_labels, test_predictions)
print(f'Test Accuracy: {test_accuracy * 100:.2f}%')


Using cache found in /root/.cache/torch/hub/pytorch_vision_v0.10.0
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


NameError: name 'learning_rate' is not defined

In [ ]:
!pip install Pillow